In [1]:
import pandas as pd
from sqlalchemy import create_engine
from dotenv import load_dotenv
import os
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression# pattern
from sklearn.ensemble import RandomForestClassifier


In [2]:
engine = create_engine(os.getenv("DB_URL"))

In [3]:
df = pd.read_sql('select * from ballon_dor.ballon_dor_features', engine)
df.shape

(438, 14)

In [4]:
train = df[df.final_rank.notna()].copy()
test = df[df.final_rank.isna()].copy()

train.shape, test.shape

((408, 14), (30, 14))

In [5]:
train["is_winner"] = (train["final_rank"] == 1).astype(int)

In [6]:
train["has_stats"] = train["goals"].notna().astype(int)
train["has_stats"].value_counts()

has_stats
1    396
0     12
Name: count, dtype: int64

In [7]:
median_goals = train["goals"].median()      # median ignores NaN by default
train["goals"] = train["goals"].fillna(median_goals)

median_minutes = train["minutes"].median()
train["minutes"] = train["minutes"].fillna(median_minutes)

median_assists = train["assists"].median()
train["assists"] = train["assists"].fillna(median_assists)

In [8]:
test["has_stats"] = test["goals"].notna().astype(int)
test["has_stats"].value_counts()

has_stats
1    27
0     3
Name: count, dtype: int64

In [9]:
test["goals"] = test["goals"].fillna(median_goals)
test["minutes"] = test["minutes"].fillna(median_minutes)
test["assists"] = test["assists"].fillna(median_assists)

In [10]:
test[["goals","assists","minutes"]].isna().sum()

goals      0
assists    0
minutes    0
dtype: int64

In [11]:
train

,season,player_name,main_club,final_rank,won_league,ucl_stage,club,minutes,goals,assists,main_position,nation,tournament,intl_stage,is_winner,has_stats
0,2010,Andrés Iniesta,Barcelona,2.0,1,4,Barcelona,1905.0,1.0,3.0,MF,ESP,WC,6,0,1
1,2010,Arjen Robben,Bayern Munich,14.0,1,5,Bayern Munich,1778.0,16.0,5.0,FW,NED,WC,5,0,1
2,2010,Asamoah Gyan,Sunderland,18.0,0,0,Rennes,2079.0,13.0,2.0,FW,GHA,WC,3,0,1
3,2010,Bastian Schweinsteiger,Bayern Munich,16.0,1,5,Bayern Munich,2903.0,2.0,2.0,DF,GER,WC,4,0,1
4,2010,Carles Puyol,Barcelona,11.0,1,4,Barcelona,2712.0,1.0,1.0,DF,ESP,WC,6,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
403,2025,Serhou Guirassy,Borussia Dortmund,21.0,0,3,Dortmund,2598.0,21.0,4.0,FW,GUI,None,0,0,1
404,2025,Viktor Gyökeres,Sporting CP,15.0,0,1,None,2606.0,10.0,6.0,None,SWE,None,0,0,0
405,2025,Vinicius Júnior,Real Madrid,16.0,0,3,Real Madrid,2253.0,11.0,8.0,FW,BRA,None,0,0,1
406,2025,Virgil van Dijk,Liverpool,28.0,1,2,Liverpool,3330.0,3.0,1.0,DF,NED,None,0,0,1


In [12]:
features = ["goals", "assists", "minutes", "ucl_stage", "intl_stage", "won_league", "has_stats"]

X_train = pd.get_dummies(train[features + ["main_position"]], columns=["main_position"], drop_first=True)
X_test  = pd.get_dummies(test[features + ["main_position"]],  columns=["main_position"], drop_first=True)
X_test  = X_test.reindex(columns=X_train.columns, fill_value=0)

y_train = train["is_winner"]

In [13]:
train["is_winner"].value_counts(),

(is_winner
 0    393
 1     15
 Name: count, dtype: int64,)

In [14]:
model = RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=1)

In [15]:
model.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",200
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric

In [16]:
proba = model.predict_proba(X_test)[:,1]
proba

array([0.005, 0.005, 0.   , 0.005, 0.005, 0.03 , 0.035, 0.   , 0.05 ,
       0.   , 0.   , 0.   , 0.02 , 0.01 , 0.115, 0.005, 0.01 , 0.   ,
       0.005, 0.   , 0.02 , 0.   , 0.03 , 0.015, 0.005, 0.   , 0.   ,
       0.015, 0.   , 0.   ])

In [22]:
ranking = test[["player_name", "main_club"]].copy()
ranking["proba"] = proba
ranking.sort_values("proba", ascending=False).head(10)


,player_name,main_club,proba
422,Lamine Yamal,Barcelona,0.115
416,Harry Kane,Bayern Munich,0.050
414,Ferran Torres,Barcelona,0.035
413,Fabián Ruiz Peña,Paris Saint-Germain,0.030
430,Ousmane Dembélé,Paris Saint-Germain,0.030
428,Michael Olise,Bayern Munich,0.020
420,Khvicha Kvaratskhelia,Paris Saint-Germain,0.020
435,Vitinha,Paris Saint-Germain,0.015
431,Pau Cubarsí,Barcelona,0.015
424,Lionel Messi,Inter Miami,0.010


In [23]:
ranking.to_csv("../data/predictions_2026.csv", index=False)

In [26]:
winners = []

for seed in range(1000):
    model = RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=seed)
    model.fit(X_train, y_train)
    p = model.predict_proba(X_test)[:,1]
    position = p.argmax()
    top = test["player_name"].iloc[position]
    winners.append(top)

In [28]:
pd.Series(winners).value_counts().to_csv("../data/mc_top_counts_2026.csv")